In [13]:
import numpy as np # used for array operations
import brian2 as b2 # used for neural simulation
from brian2 import NeuronGroup, Synapses, PoissonGroup, SpikeMonitor, run, ms, Hz
from scipy.stats import poisson, binom # used for stats (mean, s.d.)
import matplotlib.pyplot as plt # used for plotting
import json # to serialize
b2.prefs.codegen.target = "numpy" # set Brian2 to use numpy backend

In [15]:
# Simulates multiple independent Poisson processes for a group of neurons
def independent_poisson_processes(num_neurons, rate, time, num_samples):
    '''
    ***independent_poisson_processes(num_neurons, rate, time, num_samples)***
    inputs
    num_neurons - The number of neurons to simulate
    rate - The firing rate of the neurons
    time - The duration of the simulation
    num_samples - The number of samples to generate

    Simulates multiple independent Poisson processes for a group of neurons, 
    generating spike trains and saving them to files.

    Notes: This function outputs spike monitors and spike trains, which are used 
    to analyze the simulated neural activity. The spike monitors record the 
    spikes generated by the neurons, while the spike trains store the timing 
    of these spikes. These outputs can be used to calculate statistics, such as 
    mean and covariance, and to visualize the simulated neural activity.

    outputs
    spike_monitors - A dictionary of spike monitors, one for each sample
    spike_trains - A dictionary of spike trains, one for each sample
    '''
    # Convert time to milliseconds to Brian2 units
    simulation_time = time * b2.ms
    
    # Create Poisson neurons and monitor their spikes
    poisson_group = b2.PoissonGroup(num_neurons, rate * b2.Hz) # represents the neurons created, firing at the specified rate with int and rate
    spike_monitors = {} # dictionary for spike monitors
    # Create a spike monitor for each sample
    spike_monitors[sample] = b2.SpikeMonitor(poisson_group) # records the spikes generated by the poissongroup with source
    
    # Create a dictionary for spike trains
    spike_trains = {}
    
    # Create a 2D array with dimensions num_neuron and spikes
    spikes = np.zeros((num_neurons, num_samples))
    
    for sample in range(num_samples):
        # create a network
        net = b2.Network(poisson_group, spike_monitors[sample])
        
        # run the network
        net.run(simulation_time)
        
        # Save the spike monitor for each sample
        np.save(f'spike_monitor_{sample}.npy', spike_monitors[sample].spike_trains())
        
        # Store the spike trains in the dictionary
        spike_trains[sample] = spike_monitors[sample].spike_trains()
        for neuron in range(num_neurons):
            spikes[neuron, sample] = len(spike_train[neuron])

        # Calculate statistics (mean, covariance, etc.) on this 2D array
        mean_spikes = np.mean(spikes, axis=1)
        covariance_spikes = np.cov(spikes)
    
    # Save the spike trains dictionary to a JSON file
    with open('spike_trains.json', 'w') as f:
        json.dump(spike_trains, f)
    
    ### Load the spike trains dictionary from the JSON file
    #with open('spike_trains.json', 'r') as f:
    #    loaded_spike_trains = json.load(f)
    
    # Save the spike trains dictionary to a numpy file
    np.save('spike_trains.npy', spike_trains)
    
    ### Load the spike trains dictionary from the numpy file
    # loaded_spike_trains_np = np.load('spike_trains.npy', allow_pickle=True)
    return spike_monitors, spike_trains, spikes, mean_spikes, covariance_spikes

In [17]:
# Example usage:
num_neurons = 10
rate = 10
time = 1000
num_samples = 100

spike_monitors, spike_trains, spikes, mean_spikes, covariance_spikes = independent_poisson_processes(num_neurons, rate, time, num_samples)

UnboundLocalError: cannot access local variable 'sample' where it is not associated with a value

In [3]:
def save_data(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f)

In [4]:
def load_data(filename):
    with open(filename, 'r') as f:
        return json.load(f)

In [5]:
# Load the spike trains dictionary from the JSON file
with open('spike_trains.json', 'r') as f:
    spike_trains = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'spike_trains.json'

In [ ]:
# sums across neurons to get population activity
def calculate_population_activity(processes):
    return np.sum(processes, axis=1)

In [ ]:
# count total spikes for each neuron in each sample
def calculate_spike_counts(processes):
    return np.sum(processes, axis=2)

In [ ]:
# sum spike counts across neurons for each sample
def calculate_total_spike_counts(spike_counts):
    return np.sum(spike_counts, axis=1)

In [ ]:
# calculate average spike count across all samples
def calculate_average_spike_count(total_spike_counts):
    return np.mean(total_spike_counts)

In [ ]:
# Find neurons with minimum spike count
def find_minimum_spike_count(spike_counts):
    minimum_spike_count = np.min(spike_counts)
    minimum_spike_count_indices = np.where(spike_counts == minimum_spike_count)
    minimum_spike_count_integers = np.arange(1, minimum_spike_count + 1)
    return minimum_spike_count, minimum_spike_count_indices, minimum_spike_count_integers

In [ ]:
def correlated_poisson_processes(num_neurons, rate, time, num_common, num_samples):
    '''
    ***correlated_poisson_processes(num_neurons, rate, time, num_common, num_samples)***
    inputs
    num_neurons - The number of neurons to simulate
    rate - The firing rate of the neurons
    time - The duration of the simulation
    num_common - The number of common neurons that will be correlated
    num_samples - The number of samples to generate

    Simulates correlated Poisson processes for a group of neurons by combining independent Poisson processes.

    Notes: This function outputs a 3D numpy array representing the correlated spike trains,
    where each element is a boolean indicating whether a spike occurred at that time step.
    The first dimension represents the sample, the second dimension represents the neuron,
    and the third dimension represents the time step.

    outputs
    correlated_procs - A 3D numpy array representing the correlated spike trains.
    ### should output spike trains and spike monitors but for correlated ###
    '''
    # Create a network with independent Poisson neurons
    # Each neuron is reprsented by a seperate PoissonGroup with rate
    net = b2.Network()
    poisson_groups = [b2.PoissonGroup(1, rate*b2.Hz) for _ in range((num_neurons + num_common) * num_samples)]
    net.add(poisson_groups)
    
    # Setting up spike monitors to record the activity of each neuron with a simulation run
    spike_monitors = [b2.SpikeMonitor(group) for group in poisson_groups]
    net.add(spike_monitors)
    net.run(time * b2.ms) # should all of the b2.ms be (t/b2.ms)*10) ????

    # Convert spike trains for Brian2 format to numpy arrays, intialize storing process, convert spike times to discrete time indicies
    spike_trains = [monitor.spike_trains()[0] for monitor in spike_monitors]
    independent_procs = np.zeros((num_samples, num_neurons + num_common, int(time)), dtype=bool)
    spike_indices_list = [[int(t / b2.ms) for t in spike_train if t < time * b2.ms] for spike_train in spike_trains]

    # Ensuring all spike trains have the same length by padding with zeros
    max_length = max(len(indices) for indices in spike_indices_list)
    padded_spike_indices_list = [indices + [0] * (max_length - len(indices)) for indices in spike_indices_list]

    # Convert padded lists to numpy array for efficient processing
    spike_indices = np.array(padded_spike_indices_list)

    # Calculate sample and neuron idicies for efficient array indexing
    sample_indices, neuron_indices = np.divmod(np.arange((num_neurons + num_common) * num_samples), num_neurons + num_common)
    for i, indices in enumerate(spike_indices):
        for index in indices:
            if index < int(time):
                independent_procs[sample_indices[i], neuron_indices[i], index] = True

    # Generate correlated processes:
    correlated_procs = np.zeros((num_samples, num_neurons, int(time)), dtype=bool) #  by combining independent processes
    correlated_procs[:, :num_common, :] = np.cumsum(independent_procs[:, :num_common, :], axis=2) # num_common neurons are correlated through cumsum
    correlated_procs[:, num_common:, :] = independent_procs[:, num_common:num_neurons, :] # remaining neurons maintain their independent firing patterns

    return correlated_procs # should this return more?

In [ ]:
# start - counting functions

# The counting functions are designed to calculate the number of spikes in a neuron or a group of neurons over time.
# These functions take in an array of spike counts and return the cumulative sum of the spike counts up to a specified time.
# The count_at_time function calculates the cumulative sum of the spike counts up to a specified time t,
# while the count1 function calculates the cumulative sum of the spike counts for a single neuron over time.
# The countall function calculates the cumulative sum of the spike counts for all neurons over time,
# and the counting_process_nd function calculates the cumulative sum of the spike counts for multiple neurons over time.

In [ ]:
# Simulates a small LIF neural network with random connectivity, driven by Poisson inputs
def simulate_lif_network(num_neurons, num_inputs, input_rate, time):
    '''
    ***simulate_lif_network(num_neurons, num_inputs, input_rate, time)***
    inputs
    num_neurons - The number of neurons to simulate
    num_inputs - The number of Poisson inputs to the network
    input_rate - The firing rate of the Poisson inputs
    time - The duration of the simulation

    Simulates a small LIF neural network with random connectivity, driven by Poisson inputs.

    Notes: This function outputs the spike times of the neurons in the network, which can be used to analyze the activity of the network.

    outputs
    spike_trains - A dictionary of spike trains, where each key is a neuron index and each value is a list of spike times
    ### should also output spike monitors keeping up with the outputs of the other two Poisson point functions ###
    '''
    # Convert time to milliseconds to Brian2 units
    simulation_time = time * b2.ms
    
    # Create LIF neurons and monitor their spikes
    neurons = b2.NeuronGroup(num_neurons, 'dv/dt = -v/(10*ms) : 1', threshold='v > 1', reset='v = 0')
    spike_monitor = b2.SpikeMonitor(neurons)
    
    # Create Poisson inputs
    inputs = b2.PoissonInput(target=neurons, target_var='v', N=num_neurons, rate=input_rate*b2.Hz, weight=1)
    
    # Create synapses between neurons with random connectivity
    synapses = b2.Synapses(neurons, neurons, model='w:1', on_pre='v += 0.1')
    synapses.connect(p=0.5)  # Random connectivity
    
    # Create a network
    net = b2.Network(neurons, inputs, spike_monitor, synapses)
    
    # Run the network
    net.run(simulation_time)
    
    # Return the spike times
    return spike_monitor.spike_trains()

In [ ]:
def count_rate(t, spike_train):
    '''
    ***count_rate(t, spike_train)***
    inputs
    t - The time at which to count the spikes
    spike_train - The spike train of a single neuron

    Counts the number of spikes in a spike train up to a given time.

    Notes: This function returns the number of spikes that occurred before or at the specified time.

    outputs
    n_spikes - The number of spikes in the spike train up to time t
    '''
    n_spikes = 0
    if t > spike_train[0]:
        n_spikes = np.count_nonzero(spike_train < t)
    return n_spikes

In [ ]:
def count_rates(t, spike_monitor):
    '''
    ***count_rates(t, spike_monitor)***
    inputs
    t - The time at which to count the spikes
    spike_monitor - A spike monitor containing the spike trains of multiple neurons

    Counts the number of spikes in each spike train of a spike monitor up to a given time.

    Notes: This function returns a list of spike counts, one for each neuron in the spike monitor.

    outputs
    spike_counts - A list of spike counts, where each element is the number of spikes in a spike train up to time t
    '''
    return [count_rate(t, spike_train/b2.ms) for spike_train in spike_monitor.spike_trains().values()]

In [ ]:
def counting_process_nd(independent_processes, num_samples, time, num_neurons_to_plot):
    counting_process_nd = [[[0 for _ in range(time)] for _ in range(num_neurons_to_plot)] for _ in range(num_samples)]
    for i in range(num_samples):
        counts = [0] * num_neurons_to_plot
        for j in range(time):
            for k in range(num_neurons_to_plot):
                if independent_processes[i][k][j] == 1:
                    counts[k] += 1
            for k in range(num_neurons_to_plot):
                counting_process_nd[i][k][j] = counts[k]
    return counting_process_nd

In [ ]:
# end - counting functions

In [ ]:
# start - plot functions

# The plotting functions are designed to visualize the spike counts and other data.
# The plot_neurons_spiking function plots the spike counts for multiple neurons over time,
# while the plot_neuron_spiking_standard_dev function plots the spike counts for a single neuron over time,
# along with the mean and standard deviation of the spike counts.
# The plot_two_neurons_against_time function plots the spike counts for two neurons over time,
# along with the cumulative sum of the spike counts for a third neuron.

In [ ]:
def plot_neurons_spiking(processes, time): # showing some examples of neurons spiking
    fig = plt.figure(figsize=(10,6))
    for i in range(len(processes[0])):
        plt.plot(processes[0][i], label=f'Neuron {i}')
    plt.xlabel('Time')
    plt.ylabel('Spike')
    plt.title('Neurons Over Time')
    plt.legend()
    plt.show()

In [ ]:
def plot_neuron_spiking_standard_dev(processes, time):
    fig = plt.figure(figsize=(10,6))
    plt.plot(processes[0][0], label='Neuron 1')
    sd = np.std(processes[0][0])
    mean = np.mean(processes[0][0])
    print(f"Mean: {mean:.2f}")
    print(f"Standard Deviation: {sd:.2f}")
    plt.fill_between(range(time), processes[0][0] - sd, processes[0][0] + sd, alpha=0.2, label='Standard Deviation')
    plt.axhline(y=mean, color='black', linestyle='--', label='Mean')
    plt.xlabel('Time')
    plt.ylabel('Spike')
    plt.title('Neuron 1 Over Time with Standard Deviation')
    plt.legend()
    plt.text(0.5, 0.9, f"Mean: {mean:.2f}, SD: {sd:.2f}", transform=plt.gca().transAxes)
    plt.show()

In [ ]:
def plot_two_neurons_against_time(processes, time):
    fig = plt.figure(figsize=(10,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(processes[0][0], processes[0][1], np.cumsum(processes[0][2]))
    ax.set_xlabel('Neuron 1')
    ax.set_ylabel('Neuron 2')
    ax.set_zlabel('Cumulative Sum of Neuron 3')
    plt.title('Two Neurons Over Time with Cumulative Sum of Third Neuron')
    plt.show()

In [ ]:
def plot_count_neuron1_vs_time(counting_process_nd, num_samples):
    plt.figure(figsize=(10,6))
    for i in range(num_samples):
        plt.plot(counting_process_nd[i][0], label=f'Sample {i}')
    mean_neuron1 = np.mean([counting_process_nd[i][0] for i in range(num_samples)], axis=0)
    plt.plot(mean_neuron1, label='Mean', color='black', linewidth=2)
    plt.xlabel('Time')
    plt.ylabel('Count of Neuron 1')
    plt.title('Count of Neuron 1 Over Time')
    plt.ylim(0, None)  # Set y-axis lower limit to 0
    plt.show()

In [ ]:
def plot_count_neuron1_vs_neuron2_vs_time(counting_process_nd, num_samples):
    fig = plt.figure(figsize=(12,8))
    ax = fig.add_subplot(111, projection='3d')
    for i in range(num_samples):
        ax.plot(counting_process_nd[i][0], counting_process_nd[i][1], range(len(counting_process_nd[i][0])))
    mean_neuron1 = np.mean([counting_process_nd[i][0] for i in range(num_samples)], axis=0)
    mean_neuron2 = np.mean([counting_process_nd[i][1] for i in range(num_samples)], axis=0)
    ax.plot(mean_neuron1, mean_neuron2, range(len(mean_neuron1)), color='black', linewidth=2)
    ax.set_xlabel('Count of Neuron 1')
    ax.set_ylabel('Count of Neuron 2')
    ax.set_zlabel('Time', rotation=90)
    ax.set_title('Count of Neuron 1 vs Count of Neuron 2 vs Time')
    ax.set_xlim(0, max([max(counting_process_nd[i][0]) for i in range(num_samples)]))
    ax.set_ylim(0, max([max(counting_process_nd[i][1]) for i in range(num_samples)]))
    ax.set_zlim(0, max([len(counting_process_nd[i][0]) for i in range(num_samples)]))
    plt.show()

In [ ]:
def plot_count1(count1_output, title='Count of Spikes Over Time (Single Neuron)'):
    plt.figure(figsize=(10,6))
    plt.plot(count1_output)
    plt.xlabel('Time')
    plt.ylabel('Count')
    plt.title(title)
    plt.show()
    print(f"Plot of {title} generated successfully.")

In [ ]:
def plot_vectorized_count1(count1_vectorized_output, title='Count of Spikes Over Time (Single Neuron, Vectorized)'):
    plt.figure(figsize=(10,6))
    plt.plot(count1_vectorized_output)
    plt.xlabel('Time')
    plt.ylabel('Count')
    plt.title(title)
    plt.show()
    print(f"Plot of {title} generated successfully.")

In [ ]:
def plot_counts(counts_all, title='Count of Spikes Over Time'):
    plt.figure(figsize=(10,6))
    for i, count in enumerate(counts_all):
        plt.plot(count, label=f'Neuron {i}')
    plt.xlabel('Time')
    plt.ylabel('Count')
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
def plot_counts_vectorized(counts_all_vectorized, title='Count of Spikes Over Time (Vectorized)'):
    plt.figure(figsize=(10,6))
    for i, count in enumerate(counts_all_vectorized):
        plt.plot(count, label=f'Neuron {i}')
    plt.xlabel('Time')
    plt.ylabel('Count')
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
def plot_covariance(covariance, time):
    plt.figure(figsize=(10, 6))
    plt.plot(time, covariance)
    plt.xlabel('Time (s)')
    plt.ylabel('Covariance')
    plt.title('Covariance between Neuron Counts 1 and 2 over Time')
    plt.show()

In [ ]:
# end - plot functions

In [ ]:
# start - stats functions

# The statistical functions are designed to calculate statistical properties of the spike counts.
# These functions take in an array of spike counts and return statistical properties such as the mean, covariance, and slope of the spike counts. 
# The calculate_mean_and_covariance function calculates the mean and covariance of the spike counts for multiple neurons
# over time, while the get_slope function calculates the slope of the spike counts for multiple neurons over time.
# The calculate_mean_with_std function calculates the mean and standard deviation of the spike counts for a single neuron over time,
# and the calculate_covariance_matrix function calculates the covariance matrix of the spike counts for multiple neurons over time.

In [ ]:
def calculate_mean_and_covariance(processes):
    means = []
    covariances = []
    for i in range(len(processes[0])):
        mean = np.mean([process[i] for process in processes])
        covariance = np.cov([process[i] for process in processes], rowvar=False)
        means.append(mean)
        covariances.append(covariance)
    return means, covariances

In [ ]:
def calculate_covariance(processes):
    num_samples, num_neurons, n_bins = processes.shape
    covariance = np.zeros((n_bins,))

    for i in range(n_bins):
        neuron1_counts = processes[:, 0, i]
        neuron2_counts = processes[:, 1, i]
        covariance[i] = np.cov(neuron1_counts, neuron2_counts)[0, 1]

    return covariance

In [ ]:
def calculate_covariance_over_neurons(covariance):
    return np.mean(covariance)

In [ ]:
def measure_counts_in_time_windows(processes, time_windows):
    num_samples, num_neurons, n_bins = processes.shape
    mean_counts = []
    covariance_counts = []

    for start, end in time_windows:
        start_idx = int(start * 1000)
        end_idx = int(end * 1000)
        window_processes = processes[:, :, start_idx:end_idx]

        mean_count = np.mean(np.sum(window_processes, axis=2))
        covariance_count = calculate_covariance(window_processes)

        mean_counts.append(mean_count)
        covariance_counts.append(covariance_count)

    return mean_counts, covariance_counts

In [ ]:
def simulate_homogenous_poisson_process(rate, simulation_time):
    import numpy as np

    # Generate Poisson process using numpy's random.poisson function
    poisson_process = np.random.poisson(rate * simulation_time)

    return poisson_process

In [ ]:
def get_slope(processes):
    slopes = []
    for i in range(len(processes[0])):
        slope = np.polyfit(range(len(processes[0][i])), processes[0][i], 1)[0]
        slopes.append(slope)
    return slopes

In [ ]:
def calculate_mean_with_std(processes):
    mean_with_std = np.mean(processes[0][0]) + np.std(processes[0][0])
    return mean_with_std

In [ ]:
def calculate_covariance_matrix(processes):
    cov_matrix = np.cov([processes[0][0], processes[0][1]])
    return cov_matrix

In [ ]:
# end - stats functions